# 准备公司事件时间线

## 目标

输出去重后的事件版本、可信可得时间、下一完整交易日与待核对原因，不计算异常收益。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"company-event-timeline\",\"identity\":\"synthetic\",\"args\":[[{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\"},{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\"},{\"id\":\"DEMO-B\",\"version\":\"v1\",\"publishedAt\":\"2025-01-06\",\"firstSeenAt\":\"2025-01-06T10:00:00+08:00\"}],[\"2025-01-03T09:30:00+08:00\",\"2025-01-06T09:30:00+08:00\",\"2025-01-07T09:30:00+08:00\"]],\"expected\":[{\"id\":\"DEMO-A\",\"version\":\"v1\",\"publishedAt\":\"2025-01-03T18:00:00+08:00\",\"firstSeenAt\":\"2025-01-03T18:02:00+08:00\",\"availableAt\":\"2025-01-03T10:02:00.000Z\",\"sessionOpen\":\"2025-01-06T09:30:00+08:00\",\"status\":\"aligned\"},{\"id\":\"DEMO-B\",\"version\":\"v1\",\"publishedAt\":\"2025-01-06\",\"firstSeenAt\":\"2025-01-06T10:00:00+08:00\",\"sessionOpen\":null,\"status\":\"needs_review\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 冻结事件定义

公告、媒体报道和公司行动生效日是不同对象。先声明本次研究使用哪一种，并保留原始链接、文件内容摘要、公司识别和修订关系。相同内容被转载，不应该自动变成多个独立事件。

### 2. 明确可得时点

示例取发布时间和首次观察时间的较晚者。只有日期的记录进入needs_review，不猜测盘前还是盘后。时间戳必须带时区；不能把UTC与上海本地时间直接按字符串排序。

### 3. 采用保守的日频映射

本教程选择可得时点之后第一个开盘的完整交易日。周五盘后公告映射到日历中的下一个交易日；盘中公告也进入下一完整交易日。这是明确的日频约定，不是交易所规则，也不适用于直接分析公告后的分钟反应。

### 4. 保留无法对齐的记录

完全相同的事件版本只保留一条，冲突版本停止处理。超出日历覆盖范围的事件标记outside_calendar，补足日历后再映射，不自动猜下一个工作日。待核对记录与已对齐记录分开，避免缺失时间被悄悄掩盖。

### 方法与假设

- 不能用工作日日历替代交易所假期、停市和证券停牌状态。
- 首次抓取时间只是本数据链的证据，不证明全市场首次获知。
- 对齐事件不构成因果识别；并发事件与样本偏差仍需后续研究。

In [ ]:
def align_events(events, session_opens):
    """Map an event to the first opening strictly after its availability."""
    from datetime import datetime, timezone
    import re

    def parse(value):
        if not isinstance(value, str) or not re.search(r"T.*(Z|[+-]\d{2}:\d{2})$", value):
            return None
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            return None

    if not session_opens or any(parse(value) is None for value in session_opens):
        raise ValueError("invalid_calendar")
    sessions = sorted(set(session_opens), key=parse)
    seen, output = {}, []
    for event in events:
        if not event.get("id") or not event.get("version"):
            raise ValueError("missing_event_identity")
        key = (event["id"], event["version"])
        if key in seen:
            if seen[key] != event:
                raise ValueError("conflicting_event_version")
            continue
        seen[key] = dict(event)
        published, observed = parse(event.get("publishedAt")), parse(event.get("firstSeenAt"))
        if published is None or observed is None:
            output.append(dict(event, sessionOpen=None, status="needs_review"))
            continue
        available = max(published, observed)
        session = next((value for value in sessions if parse(value) > available), None)
        timestamp = available.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
        output.append(dict(event, availableAt=timestamp, sessionOpen=session,
                           status="aligned" if session else "outside_calendar"))
    return output


### 运行小样本

3条虚构输入去重后为2条：DEMO-A映射到2025-01-06开盘；DEMO-B只有日期，保留为needs_review且无交易日。

In [ ]:
result = align_events(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `cn.dataset.anns_d`
- `cn.market.trade_calendar`

### 参考资料

- [MacKinlay：事件研究方法](https://www.jstor.org/stable/2729691)
- [Tushare：证券身份与上市状态](https://tushare.pro/document/2?doc_id=25)

[返回教程](https://tradingdatas.com/recipes/company-event-timeline/)